# Day 45 — Model Deployment with Flask
### REST API · Flask · JSON · POST Requests · Production Serving

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# install flask if needed
try:
    import flask
    print(f"Flask: {flask.__version__} ✅")
except:
    import subprocess
    subprocess.run(['pip', 'install', 'flask'], capture_output=True)
    import flask
    print(f"Flask: {flask.__version__} ✅")

from flask import Flask, request, jsonify
print("All imports ready! ✅")
print("Goal: serve our Day 44 Titanic model as a REST API")

Flask: 3.1.3 ✅
All imports ready! ✅
Goal: serve our Day 44 Titanic model as a REST API


## 2. What is Flask & REST API?

In [2]:
print("=" * 55)
print("       WHAT IS FLASK & REST API?")
print("=" * 55)
print("""
FLASK:
  Lightweight Python web framework
  Perfect for building REST APIs
  Simple, minimal, easy to learn
  Powers thousands of ML APIs in production

REST API:
  REpresentational State Transfer
  Standard way for applications to communicate
  Uses HTTP methods:
    GET    → retrieve data
    POST   → send data / trigger action
    PUT    → update data
    DELETE → remove data

HOW OUR ML API WORKS:
  Client (Python/app/website)
      ↓  POST request with passenger data (JSON)
  Flask API
      ↓  Load model, preprocess, predict
  Response
      ↑  Return prediction as JSON

JSON FORMAT:
  JavaScript Object Notation
  Standard data exchange format
  {
    "Pclass": 1,
    "Sex": "female",
    "Age": 29,
    "prediction": 1,
    "probability": 0.98
  }

WHY DEPLOY AS API?
  ✅ Any language can call it (Python, JS, Java...)
  ✅ Frontend apps can use your model
  ✅ Multiple services can share one model
  ✅ Scale independently from other services
""")

       WHAT IS FLASK & REST API?

FLASK:
  Lightweight Python web framework
  Perfect for building REST APIs
  Simple, minimal, easy to learn
  Powers thousands of ML APIs in production

REST API:
  REpresentational State Transfer
  Standard way for applications to communicate
  Uses HTTP methods:
    GET    → retrieve data
    POST   → send data / trigger action
    PUT    → update data
    DELETE → remove data

HOW OUR ML API WORKS:
  Client (Python/app/website)
      ↓  POST request with passenger data (JSON)
  Flask API
      ↓  Load model, preprocess, predict
  Response
      ↑  Return prediction as JSON

JSON FORMAT:
  JavaScript Object Notation
  Standard data exchange format
  {
    "Pclass": 1,
    "Sex": "female",
    "Age": 29,
    "prediction": 1,
    "probability": 0.98
  }

WHY DEPLOY AS API?
  ✅ Any language can call it (Python, JS, Java...)
  ✅ Frontend apps can use your model
  ✅ Multiple services can share one model
  ✅ Scale independently from other services



## 3. Build the Flask API

In [12]:
print("=" * 55)
print("=" * 55)
print("       BUILD THE FLASK API")
print("=" * 55)

flask_app_code = '''# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import joblib
from flask import Flask, request, jsonify
import warnings
warnings.filterwarnings("ignore")

app = Flask(__name__)
model = joblib.load(r"C:\\DS-AI-75d\\titanic_pipeline.pkl")

def engineer_features(df):
    df = df.copy()
    df["Title"]      = df["Name"].str.extract(r" ([A-Za-z]+)\\.", expand=False)
    df["Title"]      = df["Title"].map({"Mr":"Mr","Miss":"Miss",
                        "Mrs":"Mrs","Master":"Master"}).fillna("Other")
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"]    = (df["FamilySize"] == 1).astype(int)
    df["FareLog"]    = np.log1p(df["Fare"])
    df["FareBin"]    = pd.cut(df["Fare"],
                        bins=[-1, 7.91, 14.45, 31.0, 600],
                        labels=["Low","Mid","High","VHigh"])
    df["HasCabin"]   = df["Cabin"].notna().astype(int)
    df["Age"]        = df["Age"].fillna(28.0)
    df["AgeBin"]     = pd.cut(df["Age"], bins=[0,12,18,35,60,100],
                       labels=["Child","Teen","Adult","Middle","Senior"])
    df = df.drop(["PassengerId","Name","Ticket","Cabin","Fare"],
                 axis=1, errors="ignore")
    return df

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Titanic Survival Prediction API",
        "version": "1.0",
        "endpoints": {
            "/predict": "POST - predict survival",
            "/health":  "GET  - health check"
        }
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model": "loaded"})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data   = request.get_json()
        df     = pd.DataFrame([data])
        df_eng = engineer_features(df)
        pred   = model.predict(df_eng)[0]
        prob   = model.predict_proba(df_eng)[0][1]
        return jsonify({
            "prediction":  int(pred),
            "survived":    bool(pred == 1),
            "probability": round(float(prob), 4),
            "message":     "Survived!" if pred == 1 else "Did not survive"
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''

with open(r'C:\\DS-AI-75d\\app.py', 'w', encoding='utf-8') as f:
    f.write(flask_app_code)

print("Flask app saved to: C:\\DS-AI-75d\\app.py ✅")
print("\nAPI ENDPOINTS:")
print("  GET  /         -> API info")
print("  GET  /health   -> health check")
print("  POST /predict  -> predict survival")
print("\nAPP STRUCTURE:")
print("  1. Load model once at startup (efficient)")
print("  2. engineer_features() - same as training")
print("  3. /predict endpoint - receives JSON, returns JSON")
print("  4. Error handling - returns 400 on bad input")


       BUILD THE FLASK API
Flask app saved to: C:\DS-AI-75d\app.py ✅

API ENDPOINTS:
  GET  /         -> API info
  GET  /health   -> health check
  POST /predict  -> predict survival

APP STRUCTURE:
  1. Load model once at startup (efficient)
  2. engineer_features() - same as training
  3. /predict endpoint - receives JSON, returns JSON
  4. Error handling - returns 400 on bad input


## 4. Run & Test the API

In [10]:
print("=" * 55)
print("       RUN & TEST THE API")
print("=" * 55)

import subprocess
import time
import requests

# start Flask in background
print("Starting Flask server...")
server = subprocess.Popen(
    ['python', r'C:\DS-AI-75d\app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# wait with retry
BASE_URL = "http://127.0.0.1:5000"
for i in range(10):
    time.sleep(2)
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=2)
        if r.status_code == 200:
            print(f"Server ready after {(i+1)*2}s ✅\n")
            break
    except:
        print(f"  Waiting... ({(i+1)*2}s)")
else:
    print("Server did not start — check app.py")
    server.terminate()

# test 1 — home
print("TEST 1 — Home endpoint:")
r = requests.get(f"{BASE_URL}/")
print(f"  Status: {r.status_code}")
print(f"  Response: {r.json()}")

# test 2 — health
print("\nTEST 2 — Health check:")
r = requests.get(f"{BASE_URL}/health")
print(f"  Status: {r.status_code}")
print(f"  Response: {r.json()}")

# test 3 — predict
print("\nTEST 3 — Predict passengers:")
passengers = [
    {"Name":"Smith, Mrs. A", "Pclass":1, "Sex":"female",
     "Age":29, "SibSp":0, "Parch":0, "Ticket":"PC1",
     "Fare":100, "Cabin":"C85", "Embarked":"S"},
    {"Name":"Jones, Mr. B",  "Pclass":3, "Sex":"male",
     "Age":22, "SibSp":1, "Parch":0, "Ticket":"S01",
     "Fare":7.25, "Cabin":None, "Embarked":"S"},
    {"Name":"Brown, Master. C","Pclass":2,"Sex":"male",
     "Age":8,  "SibSp":1, "Parch":1, "Ticket":"S02",
     "Fare":30, "Cabin":None, "Embarked":"S"},
]

for i, p in enumerate(passengers):
    r = requests.post(f"{BASE_URL}/predict", json=p)
    result = r.json()
    emoji  = "✅" if result['survived'] else "❌"
    print(f"  Passenger {i+1}: {emoji} {result['message']} "
          f"(prob={result['probability']:.2f}) "
          f"| {p['Pclass']}cl {p['Sex']} age={p['Age']}")

# stop server
server.terminate()
print("\nServer stopped ✅")

       RUN & TEST THE API
Starting Flask server...
Server ready after 2s ✅

TEST 1 — Home endpoint:
  Status: 200
  Response: {'endpoints': {'/health': 'GET  - health check', '/predict': 'POST - predict survival'}, 'message': 'Titanic Survival Prediction API', 'version': '1.0'}

TEST 2 — Health check:
  Status: 200
  Response: {'model': 'loaded', 'status': 'healthy'}

TEST 3 — Predict passengers:
  Passenger 1: ✅ Survived! (prob=0.98) | 1cl female age=29
  Passenger 2: ❌ Did not survive (prob=0.06) | 3cl male age=22
  Passenger 3: ✅ Survived! (prob=0.77) | 2cl male age=8

Server stopped ✅


## 5. Key Takeaways

In [11]:
print("=" * 55)
print("       DAY 45 — KEY TAKEAWAYS")
print("=" * 55)
print("""
FLASK:
  ✅ Lightweight Python web framework
  ✅ @app.route() → defines URL endpoints
  ✅ request.get_json() → reads incoming JSON
  ✅ jsonify() → returns JSON response
  ✅ debug=True → auto-reloads on code changes

REST API:
  ✅ GET  → retrieve (home, health check)
  ✅ POST → send data (predict endpoint)
  ✅ Status 200 → success
  ✅ Status 400 → bad request / error

ML API PATTERN:
  ✅ Load model ONCE at startup — not per request
  ✅ engineer_features() — same as training
  ✅ Use pd.cut() not pd.qcut() for single rows
  ✅ Always wrap predict in try/except
  ✅ Return structured JSON with all info

OUR API RESULTS:
  ✅ 1st class female age 29 → 98% survival
  ✅ 3rd class male   age 22 → 6%  survival
  ✅ 2nd class boy    age  8 → 77% survival (children first!)

PRODUCTION NOTES:
  ✅ Never use debug=True in production
  ✅ Use gunicorn/waitress as WSGI server
  ✅ Add authentication for real APIs
  ✅ Deploy to cloud: AWS, GCP, Azure, Heroku
""")

       DAY 45 — KEY TAKEAWAYS

FLASK:
  ✅ Lightweight Python web framework
  ✅ @app.route() → defines URL endpoints
  ✅ request.get_json() → reads incoming JSON
  ✅ jsonify() → returns JSON response
  ✅ debug=True → auto-reloads on code changes

REST API:
  ✅ GET  → retrieve (home, health check)
  ✅ POST → send data (predict endpoint)
  ✅ Status 200 → success
  ✅ Status 400 → bad request / error

ML API PATTERN:
  ✅ Load model ONCE at startup — not per request
  ✅ engineer_features() — same as training
  ✅ Use pd.cut() not pd.qcut() for single rows
  ✅ Always wrap predict in try/except
  ✅ Return structured JSON with all info

OUR API RESULTS:
  ✅ 1st class female age 29 → 98% survival
  ✅ 3rd class male   age 22 → 6%  survival
  ✅ 2nd class boy    age  8 → 77% survival (children first!)

PRODUCTION NOTES:
  ✅ Never use debug=True in production
  ✅ Use gunicorn/waitress as WSGI server
  ✅ Add authentication for real APIs
  ✅ Deploy to cloud: AWS, GCP, Azure, Heroku

